# Vérification des P0 et P1, 12 septembre 2026

Compagnon du rapport `P0-P1-2026-09-12.md`. Toutes les cellules ont été exécutées avec Python 3 et sa bibliothèque standard. Lancer depuis la racine du dépôt ou depuis `docs/seo`. Les deux archives natives doivent être dans `~/Downloads` ; adapter `downloads` si elles ont été déplacées.

Les observations GSC sont une transcription des tableaux complets affichés dans le navigateur interne autorisé, avec les filtres indiqués dans le JSON. Elles ne constituent pas un export API. Ce notebook vérifie leur cohérence et les calculs ; il ne restaure pas les requêtes omises par Google et ne réinterroge pas le service. Les métadonnées sont un relevé public daté, pas une observation du snippet Google.

In [1]:
from pathlib import Path
import csv, hashlib, io, json, math, re, shutil, subprocess, tempfile, zipfile
from urllib.parse import parse_qs, urlparse

cwd = Path.cwd().resolve()
root = next(p for p in (cwd, *cwd.parents) if (p / 'scripts/validate-seo.mjs').is_file())
docs = root / 'docs/seo'
downloads = Path.home() / 'Downloads'
load = lambda filename: json.loads((docs / filename).read_text())
evidence = load('gsc-filtered-observations-2026-09-12.json')
baseline = evidence['baseline_reconciliation']
registry = load('seo-title-experiments-2026-09.json')['measurement_followup']
metadata = load('metadata-baseline-2026-09-12.json')
links = load('contextual-links-2026-09-12.json')
print('Période GSC :', evidence['period_start'], 'au', evidence['period_end'])
print('Fuseau des journées de performance :', evidence['performance_date_timezone'])


Période GSC : 2026-09-04 au 2026-09-09
Fuseau des journées de performance : America/Los_Angeles


## Archives natives : intégrité et nouvelles journées

In [2]:
def native_rows(archive, member):
    with zipfile.ZipFile(archive) as z:
        return list(csv.DictReader(io.StringIO(z.read(member).decode('utf-8-sig'))))

archives = {}
for version in ['baseline', 'latest']:
    path = downloads / baseline[version + '_file']
    assert hashlib.sha256(path.read_bytes()).hexdigest() == baseline[version + '_sha256']
    archives[version] = path
old = {r['Date']: r for r in native_rows(archives['baseline'], 'Graphique.csv')}
new = {r['Date']: r for r in native_rows(archives['latest'], 'Graphique.csv')}
common = old.keys() & new.keys()
assert len(common) == baseline['identical_common_days'] == 44
assert all(old[date] == new[date] for date in common)
added = sorted(new.keys() - old.keys())
assert len(added) == baseline['added_days'] == 6
assert (added[0], added[-1]) == ('2026-09-04', '2026-09-09')
assert sum(int(new[d]['Clics']) for d in added) == baseline['added_clicks'] == 67
assert sum(int(new[d]['Impressions']) for d in added) == baseline['added_impressions'] == 6181
print('Deux SHA-256 conformes ; 44 journées communes identiques ; six journées ajoutées : 67 clics / 6 181 impressions.')


Deux SHA-256 conformes ; 44 journées communes identiques ; six journées ajoutées : 67 clics / 6 181 impressions.


## Requêtes tronquées et scénario à 1 %
Le scénario reste hypothétique. Une différence entre listes de requêtes dont la composition change ne mesure pas la couverture de la période ajoutée.

In [3]:
query_sets = []
query_sums = []
for version in ['baseline', 'latest']:
    rows = native_rows(archives[version], 'Requêtes.csv')
    keys = {r['Requêtes les plus fréquentes'] for r in rows}
    assert len(rows) == len(keys) == 1000
    query_sets.append(keys)
    query_sums.append(sum(int(r['Impressions']) for r in rows))
assert len(query_sets[1] - query_sets[0]) == baseline['query_keys_entering'] == 157
assert len(query_sets[0] - query_sets[1]) == baseline['query_keys_leaving'] == 157
assert query_sums[1] - query_sums[0] == baseline['query_snapshot_impression_difference'] == 528
assert baseline['query_snapshot_difference_is_period_coverage'] is False

def page_total(version, prefix, metric):
    return sum(int(r[metric]) for r in native_rows(archives[version], 'Pages.csv')
               if r['Pages les plus populaires'].startswith(prefix))

deltas = {}
for kind in ['analysis', 'guides']:
    prefix = 'https://l0g.fr/en/' + kind + '/'
    deltas[kind] = {metric: page_total('latest', prefix, metric) - page_total('baseline', prefix, metric)
                    for metric in ['Clics', 'Impressions']}
assert deltas['analysis']['Impressions'] == 3186
assert deltas['guides']['Impressions'] == 1078
observed_clicks = sum(v['Clics'] for v in deltas.values())
denominator = sum(v['Impressions'] for v in deltas.values())
assert observed_clicks == 9 and denominator == 4264
assert math.isclose(denominator * .01, baseline['one_percent_scenario']['hypothetical_clicks'])
assert math.isclose(denominator * .01 - observed_clicks, 33.64)
print('157 requêtes entrantes et 157 sortantes : 528 ne mesure pas la couverture.')
print('Scénario : 4 264 × 1 % = 42,64 clics hypothétiques ; supplément = 33,64, sans valeur prédictive.')


157 requêtes entrantes et 157 sortantes : 528 ne mesure pas la couverture.
Scénario : 4 264 × 1 % = 42,64 clics hypothétiques ; supplément = 33,64, sans valeur prédictive.


## Grain des observations et CSV
`ALL` est un agrégat, pas une valeur de pays. Les cellules vides restent des limites de restitution ; les positions ne sont pas reconstituées.

In [4]:
pages = evidence['pages']
assert len(pages) == len({p['id'] for p in pages}) == 10
expected = []
for page in pages:
    for segment, country, device in [('all', 'ALL', 'ALL'), ('us_desktop', 'USA', 'DESKTOP')]:
        metrics = page[segment]
        filters = parse_qs(urlparse(page['source_url_' + segment]).query)
        assert filters['page'] == ['!https://l0g.fr' + page['route']]
        assert filters['start_date'] == ['20260904'] and filters['end_date'] == ['20260909']
        if segment == 'us_desktop':
            assert filters['country'] == ['usa'] and filters['device'] == ['DESKTOP']
        assert metrics['all_visible_rows_captured'] is True
        assert len(metrics['queries']) == len({q['query'] for q in metrics['queries']})
        if metrics['impressions'] == 0:
            assert metrics['position_displayed'] is None and metrics['ctr'] is None
        else:
            assert math.isclose(metrics['ctr'], metrics['clicks'] / metrics['impressions'])
        if segment == 'all':
            assert sum(q['impressions'] for q in metrics['queries']) == metrics['visible_query_impressions']
        for q in metrics['queries']:
            assert q['impressions'] > 0 and 0 <= q['clicks'] <= q['impressions']
            assert math.isclose(q['ctr'], q['clicks'] / q['impressions'])
            expected.append(('https://l0g.fr' + page['route'], q['query'], country, device,
                             q['clicks'], q['impressions'], q['ctr'], q['position_displayed']))
with (docs / 'gsc-page-query-segments-2026-09-12.csv').open(newline='') as f:
    actual = list(csv.DictReader(f))
assert len(actual) == len(expected) == 30
for row, want in zip(actual, expected):
    assert row['period_start'] == evidence['period_start'] and row['period_end'] == evidence['period_end']
    got = (row['page'], row['query'], row['country'], row['device'], int(row['clicks']),
           int(row['impressions']), float(row['ctr']), float(row['position_displayed']))
    assert got == want
segment = evidence['english_us_desktop']
assert len(segment['pages']) == 24 and len(segment['queries']) == 26
assert segment['impressions'] == 73 and segment['clicks'] == 0
assert sum(p['impressions'] for p in segment['pages']) == 73
assert sum(q['impressions'] for q in segment['queries']) == 73
assert segment['page_query_join_available_for_all_segment_pages'] is False
print('10 pages, 30 lignes de requêtes exactes ; segment EN × USA × desktop : 24 pages et 26 requêtes, 73 impressions restituées.')


10 pages, 30 lignes de requêtes exactes ; segment EN × USA × desktop : 24 pages et 26 requêtes, 73 impressions restituées.


## Marque, hors-marque visible et résidu
Les totaux viennent du graphique filtré. Les requêtes omises restent non classées. La position moyenne résiduelle est indisponible.

In [5]:
for week in evidence['weekly_brand']['weeks']:
    days = [r for d, r in new.items() if week['period_start'] <= d <= week['period_end']]
    assert len(days) == 7
    for metric, native_name in [('clicks', 'Clics'), ('impressions', 'Impressions')]:
        assert sum(int(r[native_name]) for r in days) == week['unfiltered'][metric]
        assert sum(week[k][metric] for k in ['brand', 'visible_nonbrand', 'unclassified']) == week['unfiltered'][metric]
    assert week['unclassified']['position_displayed'] is None
    for kind in ['brand', 'visible_nonbrand', 'unclassified', 'unfiltered']:
        value = week[kind]
        assert math.isclose(value['ctr'], value['clicks'] / value['impressions'])
    print(week['period_start'], week['period_end'], 'total', week['unfiltered']['clicks'], 'clics /',
          week['unfiltered']['impressions'], 'impressions ; non classé', week['unclassified']['impressions'])


2026-08-27 2026-09-02 total 53 clics / 4370 impressions ; non classé 3803
2026-09-03 2026-09-09 total 83 clics / 7431 impressions ; non classé 6597


## Titres, inspections et maillage
Les dates de crawl conservent le fuseau inconnu de l’affichage. Pour AI/BIS, le titre a été lu dans le HTML exploré ; la seule journée entière mesurée après ce crawl est le 9 septembre. Le maillage est local, avant publication.

In [6]:
public = {p['path']: p for p in metadata['pages']}
frozen = registry['pages']
assert len(frozen) == 9
assert sum(p['post_recrawl_impressions'] is None for p in frozen) == 8
for page in frozen:
    assert public[page['route']]['status'] == 200
    assert public[page['route']]['title'] == page['observed_title']
    assert public[page['route']]['canonical'] == 'https://l0g.fr' + page['route']
ai = next(p for p in pages if p['id'] == 'ai-bis')
assert ai['post_recrawl_impressions'] == 20
assert ai['post_recrawl_window']['start'] == ai['post_recrawl_window']['end'] == '2026-09-09'
assert ai['inspection']['crawled_title'] == public[ai['route']]['title']
assert all(p['inspection']['indexed'] and p['inspection']['google_canonical'] == 'inspected_url' for p in pages)
assert links['status'] == 'local_not_deployed'
content = sorted(p for folder in ['guides-en', 'posts-en'] for p in (root / 'src/content' / folder).rglob('*')
                 if p.suffix in ['.md', '.mdx'])
for target in links['targets']:
    found = {str(p.relative_to(root)) for p in content if target['target'] in p.read_text()}
    assert found == set(target['contextual_source_pages'])
    assert len(found) == target['count'] and 3 <= len(found) <= 5
assert [t['count'] for t in links['targets']] == [3, 3, 3, 3, 3, 4]
print('Neuf titres conformes au relevé public. Huit cohortes post-recrawl indisponibles ; AI/BIS : 20 impressions.')
print('Six cibles : 3, 3, 3, 3, 3 et 4 sources contextuelles distinctes.')


Neuf titres conformes au relevé public. Huit cohortes post-recrawl indisponibles ; AI/BIS : 20 impressions.
Six cibles : 3, 3, 3, 3, 3 et 4 sources contextuelles distinctes.


## Contrôle négatif du gel
Les modifications suivantes sont réalisées uniquement dans une copie temporaire, supprimée ensuite. Le contrôle doit accepter le contenu actuel et refuser chaque changement de titre suivi.

In [7]:
with tempfile.TemporaryDirectory(prefix='l0g-title-freeze-') as name:
    target=Path(name)
    for directory in ['src/content/posts','src/content/posts-en','src/content/guides','src/content/guides-en']:
        shutil.copytree(root/directory,target/directory)
    for filename in ['src/lib/seo.ts','scripts/validate-seo.mjs','docs/seo/seo-title-experiments-2026-09.json','src/pages/en/start/index.astro']:
        dest=target/filename;dest.parent.mkdir(parents=True,exist_ok=True);shutil.copyfile(root/filename,dest)
    (target/'node_modules').symlink_to(root/'node_modules',target_is_directory=True)
    def run():return subprocess.run(['node',str(target/'scripts/validate-seo.mjs')],capture_output=True,text=True)
    baseline=run();assert baseline.returncode==0,baseline.stderr
    cases=[
        ('cofer','src/content/guides-en/read-central-bank-fx-reserves-cofer.mdx','COFER: reading central bank FX reserves | l0g'),
        ('productivity','src/content/posts-en/us-productivity-real-pay-lags.md','US Productivity Q2 Revised: +1.4%, Real Pay −3.3% | l0g'),
        ('taiwan','src/content/posts-en/taiwan-life-insurers-724-billion-currency-risk.md','Taiwan life insurers: $724bn abroad and FX risk | l0g'),
        ('start','src/pages/en/start/index.astro','l0g in English: macro, credit and financial risk | l0g'),
        ('bessent','src/content/posts-en/bessent-yield-threshold-30-year-treasury.md','Treasury buybacks: why $4bn is not a Bessent put | l0g'),
    ]
    for id,filename,title in cases:
        path=target/filename;original=path.read_text();assert title in original
        path.write_text(original.replace(title,'Different title still requiring measurement | l0g',1))
        try:
            result=run();assert result.returncode!=0,id
            assert f'Title suivi modifié avant décision: {id}' in result.stderr,(id,result.stderr)
        finally:path.write_text(original)
    proof={'baseline_passed':True,'mutations_rejected':[x[0] for x in cases],'isolation':'Temporary copy; working content not mutated'}
    print(json.dumps(proof))


{"baseline_passed": true, "mutations_rejected": ["cofer", "productivity", "taiwan", "start", "bessent"], "isolation": "Temporary copy; working content not mutated"}


## Limites de validation

Ces contrôles prouvent la cohérence des fichiers et le comportement du gel local. Ils ne prouvent ni un effet causal du titre sur le CTR, ni la présence du nouveau titre dans chaque résultat Google, ni la publication des liens ou des posts X. La vérification éditoriale du dépôt a aussi été exécutée : `npm run publish:check`, avec build Astro, passe avec 30 avertissements sur les corps de contenus existants. Aucun chiffre, date de frontmatter, H1 ou titre SEO de contenu n’a été changé dans cette tâche.